In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
#Expert module
class Expert(nn.Module):
    """ An MLP is a simple linear layer followed by a non-linearity i.e. each Expert """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(0.5),
        )

    def forward(self, x):
        return self.net(x)




In [5]:
zeros = torch.full_like(logits, float('-inf')) #full_like clones a tensor and fills it with a specified value (like infinity) for masking or calculations.
sparse_logits = zeros.scatter(-1, top_k_indices, top_k_logits)
sparse_logits


tensor([[[0.3336,   -inf,   -inf, 0.3287],
         [0.2189, 0.3006,   -inf,   -inf],
         [  -inf, 0.2984, 0.2075,   -inf],
         [  -inf,   -inf, 0.4399, 0.4133]]], grad_fn=<ScatterBackward0>)

In [6]:
gating_output= F.softmax(sparse_logits, dim=-1)
gating_output

tensor([[[0.5012, 0.0000, 0.0000, 0.4988],
         [0.4796, 0.5204, 0.0000, 0.0000],
         [0.0000, 0.5227, 0.4773, 0.0000],
         [0.0000, 0.0000, 0.5066, 0.4934]]], grad_fn=<SoftmaxBackward0>)

In [9]:
# First define the top k router module 
class TopkRouter(nn.Module):
    def __init__(self, n_embed, num_experts, top_k):
        super(TopkRouter, self).__init__()
        self.top_k = top_k
        self.linear = nn.Linear(n_embed, num_experts)
    
    def forward(self, mh_ouput):
        # mh_ouput is the output tensor from multihead self attention block
        logits = self.linear(mh_output)
        top_k_logits, indices = logits.topk(self.top_k, dim=-1)
        zeros = torch.full_like(logits, float('-inf'))
        sparse_logits = zeros.scatter(-1, indices, top_k_logits)
        router_output = F.softmax(sparse_logits, dim=-1)
        return router_output, indices


In [10]:
class SparseMoE(nn.Module):
    def __init__(self, n_embed, num_experts, top_k):
        super(SparseMoE, self).__init__()
        self.router = TopkRouter(n_embed, num_experts, top_k)
        self.experts = nn.ModuleList([Expert(n_embed) for _ in range(num_experts)])
        self.top_k = top_k

    def forward(self, x):
        print(f'INPUT {x.shape}')
        gating_output, indices = self.router(x)
        print(f'AFTER ROUTER {gating_output, indices}')
        final_output = torch.zeros_like(x)

        # Reshape inputs for batch processing
        flat_x = x.view(-1, x.size(-1))
        flat_gating_output = gating_output.view(-1, gating_output.size(-1))

        # Process each expert in parallel
        for i, expert in enumerate(self.experts):
            # Create a mask for the inputs where the current expert is in top-k
            expert_mask = (indices == i).any(dim=-1)
            print(f'INDICES: {indices}')
            print(f'INDICES2: {indices == i}')

            flat_mask = expert_mask.view(-1)
            print(f'EXPERT MASK: {expert_mask}')
            if flat_mask.any():
                expert_input = flat_x[flat_mask]
                expert_output = expert(expert_input)
                print(f'EXPER_INPUT: {expert_input.shape}')
                print(f'EXPER_OUTPUT: {expert_output.shape}')
                # Extract and apply gating scores
                gating_scores = flat_gating_output[flat_mask, i].unsqueeze(1)
                weighted_output = expert_output * gating_scores

                # Update final output additively by indexing and adding
                final_output[expert_mask] += weighted_output.squeeze(1)

        return final_output


In [12]:
import torch
import torch.nn as nn

#Let's test this out
num_experts = 4
top_k = 2
n_embd = 256

mh_output = torch.randn(1, 4, n_embd)  # Example multi-head attention output
sparse_moe = SparseMoE(n_embd, num_experts, top_k)
final_output = sparse_moe(mh_output)
print("Shape of the final output:", final_output.shape)


INPUT torch.Size([1, 4, 256])
AFTER ROUTER (tensor([[[0.6183, 0.3817, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.6270, 0.3730],
         [0.0000, 0.7331, 0.0000, 0.2669],
         [0.0000, 0.0000, 0.5873, 0.4127]]], grad_fn=<SoftmaxBackward0>), tensor([[[0, 1],
         [2, 3],
         [1, 3],
         [2, 3]]]))
INDICES: tensor([[[0, 1],
         [2, 3],
         [1, 3],
         [2, 3]]])
INDICES2: tensor([[[ True, False],
         [False, False],
         [False, False],
         [False, False]]])
EXPERT MASK: tensor([[ True, False, False, False]])
EXPER_INPUT: torch.Size([1, 256])
EXPER_OUTPUT: torch.Size([1, 256])
INDICES: tensor([[[0, 1],
         [2, 3],
         [1, 3],
         [2, 3]]])
INDICES2: tensor([[[False,  True],
         [False, False],
         [ True, False],
         [False, False]]])
EXPERT MASK: tensor([[ True, False,  True, False]])
EXPER_INPUT: torch.Size([2, 256])
EXPER_OUTPUT: torch.Size([2, 256])
INDICES: tensor([[[0, 1],
         [2, 3],
         [1, 3]